In [ ]:
import seaborn as sns
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
data = pd.read_csv('Student_performance_data _.csv')

In [ ]:
# The data contains comprehensive information on 2,392 high school students, detailing their demographics, study habits, 
# parental involvement, extracurricular activities, and academic performance. The target variable is GradeClass, but I've
# chosen the GPA as the target variable in this model build. Here is also the link to the dataset: 
# https://www.kaggle.com/datasets/rabieelkharoua/students-performance-dataset

In [ ]:
data

In [ ]:
data.describe()

In [ ]:
# I'll split the data to a target variable y (GPA) and the remaining predictors to a list X (excluding StudentID and GradeClass).

In [ ]:
y = data['GPA']

In [ ]:
y

In [ ]:
X = data.iloc[:,1:13]

In [ ]:
X

In [ ]:
# I want to point out already from the beginning, that the correlation matrix shows a strong negative relation between GPA and
# Absences. It can be, thus, predicted that it will later belong to one of the most important predictors/features. 
# Other correlations are much closer to 0.

In [ ]:
data.corr()

In [ ]:
sns.heatmap(data.corr())

In [ ]:
from sklearn.model_selection import train_test_split

In [ ]:
# I'll split the data to a train and a test set (0.8/0.2).

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)


In [ ]:
from sklearn.ensemble import ExtraTreesRegressor
from sklearn.model_selection import cross_val_score

In [ ]:
# I'll build the model with ExtraTreesRegressor, since there are quite many predictors and it is easy to extract the most
# important ones with .feature_importances_. I will test and compare the model with all predictors and a subset of five most
# important ones.

In [ ]:
reg = ExtraTreesRegressor(n_estimators=100, random_state=0, n_jobs=1).fit(X_train, y_train)


In [ ]:
reg.score(X_train,y_train)

In [ ]:
cross_val_score(reg,X_train,y_train,cv=5).mean()

In [ ]:
reg.score(X_test,y_test)

In [ ]:
# The scores are very good, so basically it is not necessary to drop the least important features. I'll do it nonetheless for
# the sake of comparison. Before that though, I'll tune the parameters of the model to select the best subset for prediction.

In [ ]:
from sklearn.model_selection import GridSearchCV

In [ ]:
# A compact grid keeps tuning practical while covering tree depth, split size, and ensemble size.
tuned_parameters = [{
    'max_depth': [None, 5, 10],
    'min_samples_split': [2, 5],
    'n_estimators': [100, 300],
}]


In [ ]:
tune_reg = GridSearchCV(reg, tuned_parameters, cv=3, n_jobs=-1)


In [ ]:
tune_reg.fit(X_train, y_train)

In [ ]:
optimal_reg=tune_reg.best_estimator_

In [ ]:
optimal_reg

In [ ]:
optimal_reg.score(X_train,y_train)

In [ ]:
tune_reg.best_score_


In [ ]:
optimal_reg.score(X_test,y_test)

In [ ]:
# The scores dropped a bit from the original model.

In [ ]:
reg.feature_importances_

In [ ]:
# To make it easier to grasp, the feature importances will be plotted.

In [ ]:
barplot=plt.figure(figsize=(10,5))
plt.stem(reg.feature_importances_)
ax=plt.gca()
ax.set_xticks(np.arange(12))
ax.set_xticklabels(X.columns,rotation=90)

In [ ]:
# The most important feature (Absences) plotted against the GPA shows a very linear negative connection, as aforementioned.

In [ ]:
sns.regplot(y=y,x=X['Absences'],line_kws={"color": "red"})

In [ ]:
# I'll select the five most important features and try to optimize the new ExtraTreesRegressor with them.

In [ ]:
features = X.columns[reg.feature_importances_.argsort()[-5:][::-1]].tolist()
features


In [ ]:
# Important to note, Absences was -0.92, ParentalSupport 0.19, StudyTimeWeekly 0.18 Tutoring 0.145, and Extracurricular 0.09 in 
# the correlation matrix. Therefore, according to the correlation matrix, all but Absence affect GPA positively.

In [ ]:
X[features]

In [ ]:
Xopt_train, Xopt_test = X_train[features], X_test[features]
yopt_train, yopt_test = y_train, y_test


In [ ]:
reg_opt = ExtraTreesRegressor(n_estimators=100, random_state=0, n_jobs=1).fit(Xopt_train, yopt_train)


In [ ]:
reg_opt.score(Xopt_train,yopt_train)

In [ ]:
from sklearn.feature_selection import SelectFromModel
from sklearn.pipeline import make_pipeline

selected_model = make_pipeline(
    SelectFromModel(
        ExtraTreesRegressor(n_estimators=100, random_state=0, n_jobs=1),
        threshold=-np.inf,
        max_features=5,
    ),
    ExtraTreesRegressor(n_estimators=100, random_state=0, n_jobs=1),
)
selected_cv = cross_val_score(selected_model, X_train, y_train, cv=5, n_jobs=-1).mean()
selected_cv


In [ ]:
reg_opt.score(Xopt_test,yopt_test)

In [ ]:
# The scores dropped a bit from the two earlier models. I'll plot the real and predicted GPA on both models to demonstrate their
# accuracy.

In [ ]:
cv = cross_val_score(reg, X_train, y_train, cv=5, n_jobs=-1).mean()


In [ ]:
optimal_cv = tune_reg.best_score_


In [ ]:
cv_opt = selected_cv


In [ ]:
sns.regplot(x=y_test, y=reg.predict(X_test), line_kws={"color": "red"})
plt.xlabel('True GPA')
plt.ylabel('Predicted GPA')
plt.title('ExtraTreesRegressor, held-out test R²=%3.2f' % reg.score(X_test, y_test))


In [ ]:
sns.regplot(x=y_test, y=optimal_reg.predict(X_test), line_kws={"color": "red"})
plt.xlabel('True GPA')
plt.ylabel('Predicted GPA')
plt.title('Tuned ExtraTreesRegressor, held-out test R²=%3.2f' % optimal_reg.score(X_test, y_test))


In [ ]:
sns.regplot(x=yopt_test, y=reg_opt.predict(Xopt_test), line_kws={"color": "red"})
plt.xlabel('True GPA')
plt.ylabel('Predicted GPA')
plt.title('Five-feature ExtraTreesRegressor, held-out test R²=%3.2f' % reg_opt.score(Xopt_test, yopt_test))


In [ ]:
# The original ETR seems to be about 0.02 points better than the optimized one and about the same as the tuned parameters model. 
# This time, the full ETR-model and tuned parameters take the win.

In [ ]:
# To have some fun and to compare the earlier models with Neural Networks, I'll create two ANN models, which regress the GPA 
# with all the features and with the five most important ones.

In [ ]:
from sklearn.neural_network import MLPRegressor
from sklearn.preprocessing import StandardScaler


In [ ]:
nnreg = make_pipeline(StandardScaler(), MLPRegressor(hidden_layer_sizes=(10, 6), max_iter=500, random_state=0)).fit(X_train, y_train)


In [ ]:
nnreg.score(X_train,y_train)

In [ ]:
cross_val_score(nnreg, X_train, y_train, cv=5, n_jobs=-1).mean()


In [ ]:
nnreg.score(X_test,y_test)

In [ ]:
nnreg_opt = make_pipeline(StandardScaler(), MLPRegressor(hidden_layer_sizes=(10, 6), max_iter=500, random_state=0)).fit(Xopt_train, yopt_train)


In [ ]:
nnreg_opt.score(Xopt_train,yopt_train)

In [ ]:
cross_val_score(nnreg_opt, Xopt_train, yopt_train, cv=5, n_jobs=-1).mean()


In [ ]:
nnreg_opt.score(Xopt_test,yopt_test)

In [ ]:
nnreg_cv = cross_val_score(nnreg, X_train, y_train, cv=5, n_jobs=-1).mean()


In [ ]:
nnreg_opt_cv = cross_val_score(nnreg_opt, Xopt_train, yopt_train, cv=5, n_jobs=-1).mean()


In [ ]:
sns.regplot(x=y_test, y=nnreg.predict(X_test), line_kws={"color": "red"})
plt.xlabel('True GPA')
plt.ylabel('Predicted GPA')
plt.title('Scaled MLPRegressor, held-out test R²=%3.2f' % nnreg.score(X_test, y_test))


In [ ]:
sns.regplot(x=yopt_test, y=nnreg_opt.predict(Xopt_test), line_kws={"color": "red"})
plt.xlabel('True GPA')
plt.ylabel('Predicted GPA')
plt.title('Scaled five-feature MLPRegressor, held-out test R²=%3.2f' % nnreg_opt.score(Xopt_test, yopt_test))


In [ ]:
# Thus far, the NN with all the features has had the best cv and test scores of every model. The every-feature ANN seems to be
# the best predictor, although the 5-feature ANN scored very well too.

In [ ]:
import statsmodels.api as sm

In [ ]:
X = sm.add_constant(X)

In [ ]:
ordinary_reg = sm.OLS(y, X).fit()

In [ ]:
print(ordinary_reg.summary())

In [ ]:
# To conclude the model building, a linear regressor worked very well this time to predict GPA. Absences was the most important
# predictor in the data, but it is possible that in other data sets it would not achieve such significance. ANNs scored the best
#, but they require also more computing power, i. e. give or take.